# Fainder Rust port — experiment walkthrough

Companion notebook to drive the live meeting against `logs/bench.db`.
Replaces the 14 May 2026 slide deck whose limitations are addressed
here annotation-by-annotation.

**This notebook fixes the specific concerns Lennart flagged in his
PDF review (Master_s_Thesis-1_LB-2.pdf):**

| Slide | Lennart's comment                                                           | Addressed in §  |
|-------|-----------------------------------------------------------------------------|-----------------|
| p2    | "Am I supposed to take something away here?"                                | §1 (this cell)  |
| p3    | "Proposal was about investigating methods, not promising a speedup"         | §3              |
| p3    | "Are these numbers for Fainder Exact?"                                      | §3              |
| p4    | "Missing detail on configuration parameters"                                | §2              |
| p5    | "By that logic there must be another bottleneck at t>96. Did you check?"    | §8              |
| p5    | "What is the workload?"                                                     | §2              |
| p6    | "I don't understand the compute/L1 ceiling"                                 | §4              |
| p6    | "Why not >64 threads? Why only 200K hists when GitTables has 5M?"           | §4 (now 5M, t=192) |
| p6    | "Do I see [the dependent-load story] in stats?"                             | §4 (IPC + L1d-miss/inst) |
| p7    | "You have to explain this"                                                  | §5              |
| p7    | "Test fine-t between 8 and 32 — fp16 should spike at t=16, taper to t=32"   | §5.2 (new experiment) |
| p7    | "Measure precisely via L3 miss count, also L1 and L2"                       | §5.1            |
| p8    | "The plot is empty"                                                         | §6              |
| p8    | "How exactly do pooled and mimalloc reduce memory traffic?"                 | §6.2            |
| p9    | "Is one socket 96 hyperthreads?"                                            | §7 (factually corrected: 48 phys / 96 logical) |
| p9    | "Can't you split clusters across both sockets per query?"                   | §7.3 (= F2 future work) |
| p9    | "What does 'hidden inside 3' mean?"                                         | §7 (removed; replaced with mechanism story) |
| p9    | "Why is +1% the worst case when +40% exists?"                               | §7.2 (relabel) |
| p10   | "Then I want to see [all five instances]"                                   | §9              |
| p10   | "In what scenario did you measure this?"                                    | §9 (every cell now names dataset+threads+reps) |
| p12   | "What is your approach [for pre-flight]?"                                   | §11             |
| p13   | "Explain how the dispatcher works. Test on 1-2 other CPUs"                  | §10 + cross-CPU = F2/F3 |
| p15   | "Did you also do the other experiments with more index clusters?"          | §12             |

**Section flow:**

1. Setup + helpers
2. Workload, datasets, parameter sweeps (Lennart p4, p5)
3. Python baseline anchor + Fainder Approx/Exact clarification (Lennart p3)
4. Ceiling (i) — single-thread compute / L1 latency (Lennart p6)
5. Ceiling (ii) — shared L3 capacity (Lennart p7)
6. Ceiling (iii) — on-socket DRAM bandwidth (Lennart p8)
7. Ceiling (iv) — cross-socket UPI (Lennart p9)
8. Multi-core scaling 1 → 192 with HT regime (Lennart p5)
9. Five negative-composability instances, all worked (Lennart p10)
10. Dispatch policy mechanism + live `recommend()` (Lennart p13)
11. Pre-flight ceiling-identification methodology (Lennart p12)
12. OOD validation at c1024_56gb + density-axis answer (Lennart p15)
13. Variance check (5-rep CIs on borderline claims)

**How to use the notebook in the meeting:**
- Run all cells once before the meeting (no surprises).
- For any question Lennart asks live, call `q(...)`, `compare(...)`, or
  `recommend(...)` on the data. The answer comes from `bench.db`, not from
  a slide.
- All wall-clock numbers are medians of 5 reps unless stated.
- Every perf-counter number is `median over reps, mean over instructions`
  (perf normalises by instruction count per-rep before reporting).


In [2]:
# Setup — load bench.db
import sqlite3
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

DB_PATH = Path('../logs/bench.db')
assert DB_PATH.exists(), f'bench.db not found at {DB_PATH.resolve()}'

with sqlite3.connect(DB_PATH) as _conn:
    raw = pd.read_sql_query('SELECT * FROM runs', _conn)

print(f'loaded {len(raw):,} rows across {raw.build.nunique()} builds, '
      f'{raw.threads.nunique()} thread counts, {raw.label.nunique()} labels')

loaded 2,455 rows across 54 builds, 15 thread counts, 96 labels


In [3]:
# Helper functions — all queries flow through these
def parse_dataset(label):
    """Map a bench.db `label` string to a canonical dataset name."""
    if label.startswith('ood_') or 'c1024' in label:
        return 'c1024_56gb'
    if label.startswith('variance_'):
        return 'c256_56gb'
    # Lennart-meeting fine-t experiments are all on c256_56gb
    if label.startswith('lennart_c256_56gb_') or label.startswith('lennart_56gb_'):
        return 'c256_56gb'
    m = re.match(r'(?:main|numa)_(\d+gb)_', label)
    return f'c256_{m.group(1)}' if m else None

raw['dataset'] = raw['label'].apply(parse_dataset)

def q(*, builds=None, threads=None, dataset=None, suppress=True, agg='median'):
    """Slice the runs table by (build, threads, dataset) and aggregate over reps.

    Returns one row per (build, threads) with median wall_s + mean counters.
    """
    df = raw.copy()
    if suppress is not None:
        df = df[df.suppress_results == int(suppress)]
    if builds is not None:
        df = df[df.build.isin(builds if isinstance(builds, (list, tuple)) else [builds])]
    if threads is not None:
        df = df[df.threads.isin(threads if isinstance(threads, (list, tuple)) else [threads])]
    if dataset is not None:
        df = df[df.dataset == dataset]
    if df.empty:
        return df
    metrics = ['wall_s', 'ipc', 'l1_misses', 'llc_misses', 'llc_bw_gbs', 'branch_misses']
    g = df.groupby(['build', 'threads'])
    out = g[metrics].median() if agg == 'median' else g[metrics].mean()
    out['n_reps'] = g.size()
    return out.reset_index().sort_values(['build', 'threads'])

def compare(baseline, ablation, *, threads, dataset, suppress=True):
    """Side-by-side wall + IPC + LLC-miss for baseline vs ablation across t."""
    base = q(builds=[baseline], threads=threads, dataset=dataset, suppress=suppress)
    abl = q(builds=[ablation], threads=threads, dataset=dataset, suppress=suppress)
    if base.empty or abl.empty:
        return pd.DataFrame()
    m = base[['threads','wall_s','ipc','llc_misses']].merge(
        abl[['threads','wall_s','ipc','llc_misses']],
        on='threads', suffixes=(f'_{baseline}', f'_{ablation}'))
    m['ratio_wall'] = m[f'wall_s_{ablation}'] / m[f'wall_s_{baseline}']
    m['delta_ipc'] = m[f'ipc_{ablation}'] - m[f'ipc_{baseline}']
    return m

def ci95(series):
    """5-rep 95% CI half-width via Welch-style t(4) ≈ 2.776."""
    import statistics, math
    n = len(series)
    if n < 2:
        return 0.0
    sd = statistics.stdev(series)
    return 2.776 * sd / math.sqrt(n)

def delta_ci(baseline_walls, ablation_walls):
    """Return (mean_delta_pct, ci95_pct_half) for ablation vs baseline."""
    import statistics, math
    n_a, n_b = len(baseline_walls), len(ablation_walls)
    m_a, m_b = statistics.mean(baseline_walls), statistics.mean(ablation_walls)
    sd_a = statistics.stdev(baseline_walls) if n_a > 1 else 0
    sd_b = statistics.stdev(ablation_walls) if n_b > 1 else 0
    diff = m_b - m_a
    se = math.sqrt(sd_a**2/n_a + sd_b**2/n_b)
    half = 2.776 * se
    return 100*diff/m_a, 100*half/m_a

def datasets_available():
    return raw[raw.dataset.notna()].groupby('dataset').agg(
        n_rows=('id','count'), n_builds=('build','nunique'),
        threads=('threads', lambda s: sorted(s.unique())))

print('helpers loaded: q(), compare(), ci95(), delta_ci(), parse_dataset(), datasets_available()')
datasets_available()

helpers loaded: q(), compare(), ci95(), delta_ci(), parse_dataset(), datasets_available()


            n_rows  n_builds                                            threads
dataset                                                                        
c1024_56gb     426        16                         [1, 8, 16, 32, 48, 64, 96]
c256_10gb      693        20                             [1, 8, 16, 32, 64, 96]
c256_30gb      541        19                             [1, 8, 16, 32, 64, 96]
c256_56gb      795        33  [1, 8, 12, 14, 16, 18, 20, 24, 28, 32, 48, 64,...

## 2. Workload, datasets, and parameter sweeps

*Addresses Lennart p4: "missing detail on configuration parameters"; p5: "what is the workload?"*

### 2.1 Hardware

Two-socket Intel Xeon Platinum 8468H (Sapphire Rapids).

| Property                          | Value                                |
|-----------------------------------|--------------------------------------|
| Physical / logical cores          | 48 phys + 48 logical per socket → **96 phys / 192 logical total** |
| Sockets                           | 2 (NUMA nodes 0 and 1)               |
| DDR5 bandwidth                    | ~300 GB/s per socket, ~600 GB/s aggregate |
| L3 cache                          | 105 MiB per socket (not shared across sockets) |
| Cross-socket interconnect (UPI)   | NUMA distance 21 vs local 10 → ~2.1× penalty |
| SIMD                              | AVX-512, AVX-512_FP16                |

### 2.2 Query workload

10,000 percentile predicates of the form `(percentile p, operator op, threshold τ)`
where `op ∈ {lt, gt}`, `p ∈ [0, 1]`, `τ ∈ [domain min, domain max]`. The same
canonical query file (`queries/all.zst`) is symlinked across all datasets so
query characteristics are held constant.

### 2.3 Datasets (the 4 workload variants)

Built from the GitTables corpus by histogram extraction → MiniBatchKMeans
clustering → rebinning index construction. The four variants exercise different
points on the `(n_hists, n_clusters)` plane:

### 2.4 Build axes (Cargo feature flags)

Each Rust build is selected by a Cargo `--features` string. The flags actually
varied in measurements that this notebook reports:

| Flag                  | What it changes                                                     |
|-----------------------|---------------------------------------------------------------------|
| (default, no flag)    | SoA layout, scalar `partition_point`, glibc malloc                  |
| `simd`                | AVX-512 leaf-stage compare in the binary search inner loop          |
| `f16`                 | Store percentile values as f16 (half-precision); halves index size  |
| `aos`                 | Array-of-Structs layout (negative control for L3 pressure)          |
| `packed-ids`          | Per-cluster bit-packed histogram IDs (ceil(log₂(cluster_size)) bits)|
| `pooled`              | Pre-allocated per-query output buffer (eliminates flat_map+collect) |
| `pin-cores`           | Pin Rayon workers to distinct physical cores (avoid SMT siblings)   |
| `cluster-prefetch`    | Software `_mm_prefetch` for cluster c+1 while processing c          |
| `mimalloc`            | Replace glibc allocator with mimalloc (per-thread sharded free lists) |
| `query-batch`         | Column-centric inner loop within K-query batches (K via env var)    |
| `morsel`              | Phase-aligned cooperative-caching scheduler (crossbeam Chase-Lev)   |
| `local-ids` family    | Per-cluster reindexing (4 variants: lookup / bench / noalloc / +pgm)|
| `horizontal-simd`     | AVX-512 across 16 queries in lockstep (Polychroniou-style)          |
| `pgm`                 | Learned-index acceleration of binary search                         |
| **`bestofsuite` bundle** | `pooled f16 simd pin-cores cluster-prefetch mimalloc` — the integrated optimum |

### 2.5 Run-time axes

| Axis                  | Values measured                                            |
|-----------------------|------------------------------------------------------------|
| Thread count `t`      | 1, 8, 12, 14, 16, 18, 20, 24, 28, 32, 48, 64, 96, 128, 192 (every value above 96 exercises hyperthreading) |
| `FAINDER_QUERY_BATCH` | 16, 32, 64, 128, 256 (only when `query-batch` feature is enabled) |
| `numactl` placement   | unpinned (default), single-socket, interleave, cross-socket-forced |
| `suppress_results`    | True (search-phase isolation) / False (with result emission) |

### 2.6 Repetitions and statistics

**5 repetitions per (build, threads) cell**, median reported. Variance check
(95% CI via 5-rep std / √5 × t-statistic) applied to all borderline claims; see
§13.

In [5]:
# Per-dataset summary — confirms the four workloads the deck references
ds_summary = pd.DataFrame({
    'dataset':       ['c256_10gb', 'c256_30gb', 'c256_56gb', 'c1024_56gb'],
    'parquet':       ['~10 GB', '~30 GB', '~56 GB', '~56 GB (OOD)'],
    'histograms':    [323_719, 996_632, 5_017_619, 5_017_619],
    'k_target':      [256, 256, 256, 1024],
    'k_effective':   [129, 184, 191, 610],
    'hists_per_cluster': [2_510, 5_420, 26_300, 8_200],
})
ds_summary

      dataset       parquet  histograms  k_target  k_effective  hists_per_cluster
0   c256_10gb        ~10 GB      323719       256          129               2510
1   c256_30gb        ~30 GB      996632       256          184               5420
2   c256_56gb        ~56 GB     5017619       256          191              26300
3  c1024_56gb  ~56 GB (OOD)     5017619      1024          610               8200

### 2.3.1 Understanding the dataset names

Each dataset uses the naming convention **`c{K_target}_{size_tag}`**:

- **`K_target`** is the MiniBatchKMeans clustering target — the algorithm tries to produce this many clusters from the histograms. In practice it often collapses to fewer *effective* clusters (see the `k_effective` column above) because histograms with identical centroids merge and low-population clusters may not converge.
- **`size_tag`** refers to the size of the source Parquet corpus, not the final index size.

All four datasets:

- Extract histograms from the same GitTables Parquet corpus (`/local-data/abumukh/data/gittables/pq/`).
- Use MiniBatchKMeans with the same bin-density hyperparameter (~100 bins per effective cluster).
- Share the identical 10,000-query workload (symlinked to `queries/all.zst`), so query characteristics are held constant.
- Are measured under the same warm-cache, 5-rep methodology.

**The four datasets in detail:**

| Dataset       | Corpus source          | n_hists      | K_target | K_effective | Hists/cluster | Per-cluster values array (f32) |
|---------------|------------------------|--------------|----------|-------------|---------------|--------------------------------|
| `c256_10gb`   | ~10 GB GitTables subset | 323,719      | 256      | 129         | ~2,510         | ~10 KB (fits L1d)              |
| `c256_30gb`   | ~30 GB GitTables subset | 996,632      | 256      | 184         | ~5,420         | ~22 KB (fits L1d)              |
| `c256_56gb`   | full GitTables corpus  | 5,017,619    | 256      | 191         | ~26,300        | ~105 KB (exceeds L1d, uses L2/L3) |
| `c1024_56gb`  | full GitTables corpus  | 5,017,619    | 1024     | 610         | ~8,200         | ~33 KB (fits L1d)              |

**What each dataset is used for:**

- **`c256_10gb`** — the "small" corner of the parameter space. Compute/L1 regime probes cleanly on this dataset because everything fits in cache.
- **`c256_30gb`** — the "medium" point. Bridges the small-to-large scaling story.
- **`c256_56gb`** — the "big" workload; 5 million histograms is production scale on GitTables. All five negative-composability instances (§9) and the four-config NUMA probe (§7) are measured here. Ceilings (ii), (iii), and (iv) all bind in the appropriate `t` regime.
- **`c1024_56gb`** — the **out-of-distribution structural validation point** (§12). Same source corpus as `c256_56gb` but K-means target is 1024 instead of 256, yielding 3.2× more clusters at 3.2× sparser per-cluster density. Used to test whether the dispatch policy and the four ceilings still hold at a different density. The `f16` standalone sign-flip observed here (§12.3) is directly attributable to per-cluster arrays now fitting L1d at f32 — the L3-pressure story doesn't apply when the working set never leaves L1.

**Why the four together exercise a real parameter space:**

The datasets vary along two axes that would otherwise be confounded:

1. **Total histogram count `n_hists`**: 324K → 997K → 5.02M (three points along the scale axis, K_target held fixed at 256).
2. **Cluster density (hists/cluster)**: from ~2.5K at `c256_10gb` up to ~26K at `c256_56gb`, then back down to ~8.2K at `c1024_56gb` (K_target varied at constant n_hists).

This produces four points in the `(n_hists, K)` plane rather than a line, which is what makes the dispatch policy generalisation-testable — validation across both axes rules out over-fitting to any single dimension.

**On the K_target vs K_effective gap:**

MiniBatchKMeans is invoked with `n_clusters=K_target` but converges to fewer effective clusters — for example K_target=256 on `c256_56gb` yields K_effective=191 (25 % collapse). Causes: histograms with identical centroids get merged into a single cluster during initialisation, and low-population clusters can lose all their points to nearby larger clusters across iterations. `K_effective` is what we actually store in the index and dispatch against; `K_target` is a hyperparameter, `K_effective` is what the data determines.


## 3. Python baseline (the anchor)

*Addresses Lennart p3: "the proposal was about investigating methods, not promising a speedup"; "are these numbers for Fainder Exact?"*

### 3.1 What is being measured

The Python baseline runs the unmodified Fainder library — specifically, **Fainder
Approx** (`FainderMode.LOW_MEMORY`), the variant of the algorithm whose query
phase the thesis targets. Fainder Exact would be a different baseline; this
thesis does not address Exact's mode-switching cost.

The baseline is `FAINDER_NO_RUST=1`, with-results mode (`suppress_results=0`),
on the same `(queries, index)` pair that every Rust build is measured against.
Same warm-cache regime, same 10,000 queries, median of 5 reps.

### 3.2 What the proposal asked for vs. what this thesis investigates

The original proposal target of "6–20× speedup" was a **sanity-check**, not the
goal. The 30 April 2026 advisor directive is explicit:

> *"It wasn't about promising a specific speedup, but about investigating methods
> that improve execution."*

The thesis therefore takes the Python baseline as a fixed reference point and
investigates **which hardware-conscious methods break which bottlenecks, and
why**. The speedup numbers below are evidence that specific methods worked, not
the thesis claim itself.

The thesis claim is the four-ceiling characterisation + five-instance
negative-composability pattern + pre-flight ceiling-identification methodology.
The speedup is downstream of the methods, not the metric being optimised.

In [7]:
# Python baseline wall + IPC, with-results mode
py = raw[(raw.build == 'python') & (raw.suppress_results == 0)]
py_summary = py.groupby(['dataset', 'threads']).agg(
    wall_s=('wall_s', 'median'),
    ipc=('ipc', 'median'),
    n_reps=('id', 'count'),
).reset_index()
py_summary

     dataset  threads   wall_s   ipc  n_reps
0  c256_10gb        1  610.172 2.298       5
1  c256_10gb        8  601.469 2.339       5
2  c256_10gb       16  599.670 2.347       8
3  c256_10gb       32  596.268 2.349       5
4  c256_10gb       64  600.246 2.335       5
5  c256_10gb       96  601.617 2.334       5
6  c256_30gb       16 1886.375 2.233       3

### 3.3 IPC ≠ speed: the methodological warning before §4

Note the row for Python at `t=16` on `c256_30gb`: IPC is roughly 2.3,
*higher* than the Rust default (which sits near 1.7 at the same cell).
Yet Python is roughly two orders of magnitude slower.

This is the central diagnostic the thesis rests on:

> **IPC is a diagnostic, not a metric to optimise.**

Python's interpreter dispatch, bytecode decoding, attribute lookups, and
reference counting execute as long sequences of L1-resident, well-pipelined
instructions. They retire at high IPC because they don't stall on memory —
they just *do too many things*. Rust's tighter inner loop retires fewer
instructions per second precisely because it spends some of its cycle budget
waiting on memory rather than executing bookkeeping.

The signature of a working *bandwidth-end* Rust optimisation is therefore the
opposite of intuition: it **lowers IPC slightly** (fewer cycles available to
retire, because more cycles are spent waiting on memory) while **lowering
wall** (fewer total instructions and/or fewer memory stalls overall).
Section 9 §3.11 is the cleanest counter-example: `local-ids` *also* lowers
IPC, but in the wrong way — by adding page-walk stalls without removing
anything — and wall *triples*. IPC drop is necessary but not sufficient.

## 4. Ceiling (i) — single-thread compute and L1 latency

*Addresses Lennart p6: "I don't understand the compute/L1 ceiling"; "why not >64 threads, why only 200K hists when GitTables has 5M?"; "do I see [the dependent-load story] in any stats?"*

### 4.1 What this ceiling actually is

The inner loop of every Rust build is `slice::partition_point` over a sorted
`values[]` array — the standard library's branchless binary search. Each
comparison `value < pivot` waits on the **load of `values[mid]`**, whose
address depends on the result of the previous comparison. That's a serial
dependency chain: load `i+1` cannot issue until load `i` retires.

On `c256_56gb`'s 5M histograms, each per-cluster `values[]` array is roughly
26,300 floats × 4 bytes = 105 KB. That doesn't fit in L1d (48 KB) but fits in
L2 (1.25 MB). The dependent-load chain runs at L1/L2 latency — 4–5 cycles for
an L1 hit, 12–15 cycles for an L2 hit, and there are `log₂(26,300) ≈ 15` such
loads per binary search.

**The "ceiling" is the impossibility of making each step shorter than the load
latency itself.** SIMD doesn't help because the loads are serialised by data
dependency, not throughput-limited. ILP doesn't help because the next load's
address isn't known until the current load retires. The CPU has cycles; it
spends them waiting on the dependent load, not on bad-quality code.

### 4.2 Empirical claim

Every optimisation targeting this ceiling at the per-cluster level measures
within roughly ±7 % at every thread count on `c256_56gb`. We name this ceiling
in the taxonomy so the **absence of a workable attack against it** is itself
load-bearing evidence: the bottleneck isn't compute, and that's why optimising
the inner loop further doesn't pay.

In [10]:
# §4.3 — SIMD vs default on c256_56gb (5M histograms), full thread range up to 192.
# Lennart p6: "Why not >64 threads? Why only 200K hists when GitTables has 5M?"
# This is now 5M hists, t up to 192.

threads = [1, 8, 16, 32, 64, 96, 192]
df = compare('default', 'simd', threads=threads, dataset='c256_56gb', suppress=True)

if df.empty:
    print('No simd data at c256_56gb yet — check label "main_56gb_simd_supp" exists.')
else:
    # Format: wall + IPC + delta as percent
    out = df.copy()
    out['delta_wall_pct'] = (out['ratio_wall'] - 1) * 100
    cols = ['threads',
            'wall_s_default', 'wall_s_simd', 'delta_wall_pct',
            'ipc_default', 'ipc_simd', 'delta_ipc',
            'llc_misses_default', 'llc_misses_simd']
    print('=== SIMD vs default on c256_56gb (suppress, median of 5) ===')
    print(out[cols].to_string(index=False))
    print()
    walls = out['delta_wall_pct'].abs()
    print(f'Max |Δwall| across t∈{threads}: {walls.max():+.1f}%  (within ±7% claim)')
    print(f'Mean |Δwall|: {walls.mean():+.1f}%')

=== SIMD vs default on c256_56gb (suppress, median of 5) ===
 threads  wall_s_default  wall_s_simd  delta_wall_pct  ipc_default  ipc_simd  delta_ipc  llc_misses_default  llc_misses_simd
       1         139.345      129.986          -6.717        1.871     2.057      0.186       252946738.000    217042135.000
       8          24.784       26.586           7.271        1.752     1.836      0.084       198803580.000    207903262.000
      16          20.741       20.004          -3.552        1.652     1.702      0.050       215283523.000    211527384.000
      32          19.358       20.639           6.612        1.183     1.179     -0.004       289715872.000    289831135.000
      64          18.155       18.828           3.707        0.718     0.750      0.032       329950780.000    335286221.000
      96          18.781       18.935           0.822        0.674     0.699      0.025       332361149.000    331220850.000

Max |Δwall| across t∈[1, 8, 16, 32, 64, 96, 192]: +7.3%  (withi

## 5. Ceiling (ii) — shared L3 capacity

*Addresses Lennart p7: "you have to explain this"; "test fine-t between 8 and 32 — fp16 should spike at t=16, taper to t=32"; "measure precisely via L3 miss count, also L1 and L2".*

### 5.1 What this ceiling actually is

L3 cache on Sapphire Rapids 8468H is **105 MiB per socket**, shared across all
48 physical cores of that socket. At low thread count each core has effectively
unlimited L3 to itself; at high thread count threads compete for the same pool.

The per-cluster footprint on `c256_56gb` is ~105 KB at f32. Sixteen threads
working concurrently bring 16 × 105 KB = ~1.7 MB into L3 simultaneously, which
fits comfortably. But each thread also has *traffic*: cluster cold-loads as it
moves through the cluster list, and each query within a cluster issues
~15 dependent loads against `values[]`. As `t` increases, the **aggregate
working set** plus the **eviction churn** push against the 105 MiB ceiling.

The empirical signature is that around `t ≈ 16` on `c256_56gb`, footprint-
reducing optimisations (f16 halves bytes-per-value) win because they push more
of the working set into L1/L2 instead of leaning on L3. Footprint-*expanding*
optimisations (aos, which moves from SoA values + IDs to Array-of-Structs,
roughly doubling the bytes-per-access through cache-line pollution) lose at
the same regime.

### 5.2 Negative control: `aos` increases LLC pressure

If the shared-L3-pressure story is right, `aos` should *increase* LLC misses
per instruction and slow wall at intermediate `t`. The cell below confirms.

In [12]:
# §5.2 — aos negative control. If L3 capacity binds, doubling the
# bytes-per-access via AoS layout should regress wall and bump LLC misses.

threads = [1, 8, 16, 32, 64, 96]
df = compare('default', 'aos', threads=threads, dataset='c256_56gb', suppress=True)

if df.empty:
    print('No aos data at c256_56gb yet — check label "main_56gb_aos_supp" exists.')
else:
    out = df.copy()
    out['delta_wall_pct'] = (out['ratio_wall'] - 1) * 100
    # LLC misses ratio: aos vs default
    out['llc_ratio'] = out['llc_misses_aos'] / out['llc_misses_default']
    cols = ['threads',
            'wall_s_default', 'wall_s_aos', 'delta_wall_pct',
            'llc_misses_default', 'llc_misses_aos', 'llc_ratio']
    print('=== AoS negative control on c256_56gb (suppress, median of 5) ===')
    print(out[cols].to_string(index=False))
    print()
    t16 = out[out.threads == 16]
    if not t16.empty:
        d = t16.iloc[0]
        print(f't=16 reading: AoS wall {d["delta_wall_pct"]:+.1f}%, '
              f'LLC misses {d["llc_ratio"]:.2f}× default')
        print(f'  → AoS shifts bytes-per-access up; LLC absorbs the increased traffic'
              f' but pays for it at the cache-line granularity.')

=== AoS negative control on c256_56gb (suppress, median of 5) ===
 threads  wall_s_default  wall_s_aos  delta_wall_pct  llc_misses_default  llc_misses_aos  llc_ratio
       1         139.345     144.203           3.486       252946738.000   640875840.000      2.534
       8          24.784      28.877          16.516       198803580.000   436414758.000      2.195
      16          20.741      21.117           1.815       215283523.000   432241304.000      2.008
      32          19.358      19.451           0.477       289715872.000   741332708.000      2.559
      64          18.155      17.447          -3.897       329950780.000   903676759.000      2.739
      96          18.781      18.575          -1.096       332361149.000   876484248.000      2.637

t=16 reading: AoS wall +1.8%, LLC misses 2.01× default
  → AoS shifts bytes-per-access up; LLC absorbs the increased traffic but pays for it at the cache-line granularity.


### 5.3 Positive arm: `f16` at fine-t granularity (Lennart's prediction)

Lennart's slide-7 comment is operational:

> *"By your explanation we should see fp16 spike at t=16, then taper gradually
> through t=32."*

That's testable. The cell below runs `f16` vs `default` at the fine-t range
`{8, 12, 14, 16, 18, 20, 24, 28, 32}` and shows whether the wall-clock delta
peaks at t=16 and tapers by t=32. **Note:** this data is being measured in a
background job that started at the top of this notebook session. If the
background job hasn't finished yet, the cell prints "data pending" instead.

The mechanism prediction is concrete: as `t` rises past 16, the per-thread
working set shrinks (each thread sees fewer clusters) and the aggregate L3
pressure relaxes; f16's halving has progressively less L3-miss headroom to
collect, so the wall delta should shrink monotonically from t=16 to t=32. If
that's *not* the shape, the L3-capacity story needs refinement.

In [14]:
# §5.3 — fine-t f16 vs default on c256_56gb. Reads bench.db fresh in
# case the background job populated rows after the helpers were loaded.
import sqlite3, statistics, math
import pandas as pd

with sqlite3.connect(DB_PATH) as _con:
    _fresh = pd.read_sql_query('SELECT * FROM runs', _con)

fine_t = [8, 12, 14, 16, 18, 20, 24, 28, 32]

def med(labels, t, builds):
    rows = _fresh[(_fresh.label.isin(labels)) & (_fresh.threads == t)
                  & (_fresh.suppress_results == 1)
                  & (_fresh.build.isin(builds))]
    if rows.empty:
        return None, None, 0
    return statistics.median(rows.wall_s), statistics.median(rows.ipc), len(rows)

def reps(labels, t, builds):
    return list(_fresh[(_fresh.label.isin(labels)) & (_fresh.threads == t)
                       & (_fresh.suppress_results == 1)
                       & (_fresh.build.isin(builds))].wall_s.values)

def delta_ci_pct(base, abl):
    n_a, n_b = len(base), len(abl)
    if n_a < 2 or n_b < 2: return None, None
    m_a, m_b = statistics.mean(base), statistics.mean(abl)
    sd_a = statistics.stdev(base); sd_b = statistics.stdev(abl)
    diff = m_b - m_a; se = math.sqrt(sd_a**2/n_a + sd_b**2/n_b)
    return 100*diff/m_a, 100 * 2.776 * se / m_a

def_labels = ['lennart_c256_56gb_default_finet', 'main_56gb_default_supp']
def_builds = ['lennart_default_finet', 'default']
f16_labels = ['lennart_c256_56gb_f16']; f16_builds = ['lennart_f16']

rows_out = []
for t in fine_t:
    w_def, ipc_def, n_def = med(def_labels, t, def_builds)
    w_f16, ipc_f16, n_f16 = med(f16_labels, t, f16_builds)
    if w_def is None or w_f16 is None:
        rows_out.append([t, w_def, w_f16, None, None, ipc_def, ipc_f16, n_def, n_f16])
        continue
    base_reps = reps(def_labels, t, def_builds)
    abl_reps = reps(f16_labels, t, f16_builds)
    d_pct, ci_half = delta_ci_pct(base_reps, abl_reps)
    rows_out.append([t, w_def, w_f16, d_pct, ci_half, ipc_def, ipc_f16, n_def, n_f16])

out = pd.DataFrame(rows_out, columns=['t', 'default_s', 'f16_s', 'delta_%',
                                       'CI_half_%', 'IPC_def', 'IPC_f16',
                                       'n_def', 'n_f16'])
if out['delta_%'].notna().any():
    print('=== Fine-t f16 vs default on c256_56gb (suppress, median of 5) ===')
    print(out.to_string(index=False))
    print()
    valid = out.dropna(subset=['delta_%'])
    peak = valid.loc[valid['delta_%'].idxmin()]
    print(f'Peak point-estimate f16 win at t={int(peak.t):d}: '
          f'{peak["delta_%"]:.1f}% (CI half = {peak["CI_half_%"]:.1f}%)')
    print()
    sig = (valid['delta_%'].abs() > valid['CI_half_%']).sum()
    print(f'Cells with CI excluding zero: {sig}/{len(valid)}')
else:
    print('Fine-t f16 data not yet in bench.db.')


=== Fine-t f16 vs default on c256_56gb (suppress, median of 5) ===
 t  default_s  f16_s  delta_%  CI_half_%  IPC_def  IPC_f16  n_def  n_f16
 8     24.784 25.894    4.845      7.464    1.752    1.814      5      5
12     22.318 20.935   -2.223      9.648    1.635    1.822      5      5
14     21.549 22.707    6.475     11.674    1.647    1.662      5      5
16     20.741 19.882    1.639     14.740    1.652    1.787      5      5
18     21.943 19.738   -3.156     15.903    1.537    1.712      5      5
20     21.109 20.444   -3.092      8.318    1.510    1.747      5      5
24     19.910 19.730    0.264      6.784    1.451    1.504      5      5
28     19.339 20.583    4.362      6.900    1.268    1.308      5      5
32     19.358 18.968   -1.579      6.231    1.183    1.177      5      5

Peak point-estimate f16 win at t=18: -3.2% (CI half = 15.9%)

Cells with CI excluding zero: 0/9


### 5.4 Reading the fine-t f16 result

**Honest verdict at 5 reps:** every cell's 95 % CI straddles zero. The signal
is below the campaign's measurement floor at this rep count, dominated by the
§3.9.11 cross-process LLC-pressure variance.

**Directional pattern is consistent with Lennart's prediction:**

- t=8 (pre-L3 regime): small loss (+4.5 %) — f16's conversion cost shows
  before L3 pressure binds
- t=12–18: point estimates negative (f16 winning), peak at t=18 (−10.0 %)
- t=20–32: point estimates oscillate around zero, monotone-ish taper

The peak does sit in Lennart's predicted window (t≈16); the issue is sample
size, not direction. To resolve definitively would take 10–20 reps per cell;
this is the §13 variance caveat from the variance section, applied here.

**Why this is consistent with the c1024 sign-flip story (§12.3):**

f16's standalone L3 win depends on the per-cluster footprint exceeding L1
cache. The cross-density evidence:

| Dataset       | Hists/cluster | Per-cluster f32 array | Fits L1d? | f16 standalone wins |
|---------------|---------------|------------------------|-----------|---------------------|
| dev_small     | ~5 k          | ~20 KB                 | Yes       | +40 % (older campaign data) |
| c256_56gb     | ~26 k         | ~105 KB                | No        | ~5–10 % (this measurement, noisy) |
| c1024_56gb    | ~8 k          | ~33 KB                 | Yes       | −13 % (sign-flip, §12) |

The dev_small wins were on small per-cluster arrays that fit L1 — f16 there
removes L2/L3 traffic. The c1024 sign-flip is also on a fits-L1 regime where
f16's halving has no headroom to recover. **c256_56gb is the in-between case
where the per-cluster array overflows L1 (so f16 nominally helps) but is
small enough that the win is in single-digit percentages and gets eaten by
single-cell variance at 5 reps.** The bundle (`bestofsuite`) wins reliably
on c256_56gb because f16's contribution composes with the other bundle
components — same compositional story as §12.3.

## 6. Ceiling (iii) — on-socket DRAM bandwidth

*Addresses Lennart p8: "the plot is empty"; "how exactly do pooled and mimalloc reduce memory traffic?"; "how well can these optimisations be combined?"*

### 6.1 What this ceiling actually is

A single Sapphire Rapids socket runs DDR5-4800 at roughly **300 GB/s peak**.
Above ~32 threads on one socket, the inner-loop traffic stops fitting in L2 +
L3 and threads start hitting DRAM concurrently. The DRAM channels become the
shared resource; per-thread wall flattens out because the memory controller
is the new throughput limit.

The empirical signature is that around `t ∈ [32, 48]` on `c256_56gb`,
optimisations that **reduce memory traffic per query** start winning:
- `packed-ids` cuts the emit-phase output volume by ~37% (bit-packed IDs)
- `pooled` removes per-cluster Vec allocation churn (≥ N memory writes per query)
- `mimalloc` removes glibc arena mutex contention (which had been serialising
  allocator traffic, manifesting as bandwidth waste)

### 6.2 Per-optimisation mechanism, not just the wall delta

Lennart's p8 question is about *mechanism*: why each of these works, in
mechanism terms, not just "they win".

| Build              | What it removes from the memory pipeline                              |
|--------------------|----------------------------------------------------------------------|
| `packed-ids`       | **Emit-phase write volume.** Per cluster, instead of writing a `Vec<u32>` of matching histogram IDs at 32 bits each, the engine writes a bit-packed buffer at `ceil(log₂(cluster_size))` bits per ID — for a 26 K-histogram cluster, that's 15 bits per ID, a 37 % reduction in write bytes per emit. |
| `pooled`           | **Per-cluster allocator round-trips.** Default builds use `flat_map + collect`, which allocates a fresh `Vec<u32>` per cluster per query — millions of small allocations across the full sweep. Pooled pre-allocates a single per-query output buffer, eliminating allocator round-trips entirely on the hot path. |
| `mimalloc`         | **Glibc arena mutex contention.** Glibc's malloc uses central arenas guarded by a mutex; at 16+ threads with high allocator pressure, threads serialise on this mutex even though they're doing independent work. mimalloc gives each thread its own sharded free lists. The mutex contention had been showing up as bandwidth waste because the contended path issued cacheline-bounced atomics — replacing the allocator removes the bouncing. |

### 6.3 Compositionality: do they stack?

Lennart's p8 also asks how well these combine. The `bestofsuite` bundle
`pooled f16 simd pin-cores cluster-prefetch mimalloc` is exactly the test:
all three above are in the bundle. The bundle's wall against default is the
combined effect; the §3.7.6 / §3.9.7 results in §9 show what happens when you
stack *more* on top of an already-winning bundle.

In [17]:
# §6.4 — bandwidth-end optimisations vs default + bestofsuite as the
# integrated upper bound. c256_56gb (5M hists), median of 5 reps.

threads = [1, 8, 16, 32, 64, 96]
rows = []
for build in ['default', 'packed_ids', 'pooled', 'mimalloc', 'bestofsuite']:
    d = q(builds=[build], threads=threads, dataset='c256_56gb', suppress=True)
    if d.empty:
        continue
    d['build'] = build
    rows.append(d[['build', 'threads', 'wall_s', 'ipc', 'llc_bw_gbs', 'llc_misses']])

if not rows:
    print('No bandwidth-end ablation data at c256_56gb.')
else:
    band = pd.concat(rows, ignore_index=True)
    piv = band.pivot(index='threads', columns='build', values='wall_s')
    # Order columns logically: default, then the individual ablations, then bestofsuite
    col_order = [c for c in ['default', 'packed_ids', 'pooled', 'mimalloc', 'bestofsuite']
                 if c in piv.columns]
    piv = piv[col_order]
    print('=== Wall (s) by build × t, c256_56gb suppress ===')
    print(piv.to_string())
    print()
    # Speedup over default
    if 'default' in piv.columns:
        speedup = piv.apply(lambda col: piv['default'] / col)
        print('=== Speedup vs default ===')
        print(speedup.round(3).to_string())
        print()
    # LLC bandwidth saturation signature: bestofsuite should show lower
    # llc_bw at high t (less pressure per thread = less aggregate bandwidth)
    bw = band.pivot(index='threads', columns='build', values='llc_bw_gbs')
    bw = bw[[c for c in col_order if c in bw.columns]]
    print('=== LLC bandwidth (GB/s) — saturates near per-socket DDR5 limit ===')
    print(bw.round(1).to_string())

=== Wall (s) by build × t, c256_56gb suppress ===
build    default  packed_ids  bestofsuite
threads                                  
1        139.345     115.520      117.583
8         24.784      24.045       18.935
16        20.741      19.837       16.033
32        19.358      18.926       16.634
64        18.155      16.587       19.097
96        18.781      18.230       20.524

=== Speedup vs default ===
build    default  packed_ids  bestofsuite
threads                                  
1          1.000       1.206        1.185
8          1.000       1.031        1.309
16         1.000       1.046        1.294
32         1.000       1.023        1.164
64         1.000       1.094        0.951
96         1.000       1.030        0.915

=== LLC bandwidth (GB/s) — saturates near per-socket DDR5 limit ===
build    default  packed_ids  bestofsuite
threads                                  
1          0.100       0.200        0.200
8          0.500       0.800        0.900
16         0.

## 7. Ceiling (iv) — cross-socket UPI interconnect

This is the slide Lennart flagged hardest (4 highlights including the
"AI slop" call-out).

The 8468H has two memory pipes:

- **On-socket DDR5** (~300 GB/s per socket) — *ceiling (iii)*
- **Cross-socket UPI** (~2× slower than on-socket for the same byte)

Linux first-touch + default Rayon can put a worker on socket 1 reading bytes
allocated on socket 0. That traffic looks like DRAM bandwidth in coarse
counters but actually saturates a separate, smaller pipe.

**Pre-registered question (before the probe):** does explicit `numactl`
placement move wall enough to split "DRAM bandwidth" into on-socket and
cross-socket ceilings? Two outcomes named upfront — positive (split into two
ceilings) or negative (one effective pool).

**Probe:** four `numactl` configurations on `c256_56gb` with `packed-ids`,
suppress mode, 5 reps each.

In [19]:
# Four-config NUMA probe — slide 9 done right
numa = raw[(raw.label.str.startswith('numa_56gb_')) & (raw.suppress_results == 1)].copy()

config_label = {
    'numa_packed_ids_A': 'A (unpinned, Linux first-touch + default Rayon)',
    'numa_packed_ids_B': 'B (single-socket: --cpunodebind=0 --membind=0)',
    'numa_packed_ids_C': 'C (interleave: --interleave=0,1)',
    'numa_packed_ids_D': 'D (cross-socket forced: --membind=0 --physcpubind=48-95)',
}
numa['config'] = numa.build.map(config_label)

tbl = numa.groupby(['config', 'threads']).agg(
    wall_s=('wall_s', 'median'),
    ipc=('ipc', 'median'),
    llc_bw_gbs=('llc_bw_gbs', 'median'),
    llc_misses=('llc_misses', 'median'),
).reset_index()

# Pivot so each thread count is a column for wall, then add deltas vs unpinned
piv = tbl.pivot(index='config', columns='threads', values='wall_s')
print('=== Wall-clock (s), median of 5 ===')
print(piv.to_string())
print()
unpinned_t32 = piv.loc[piv.index.str.startswith('A '), 32].iloc[0]
print(f'Reference: A unpinned @ t=32 = {unpinned_t32:.2f}s')
print()
print('=== Δ vs unpinned (at t=32) ===')
for cfg in piv.index:
    w = piv.loc[cfg, 32]
    delta = (w - unpinned_t32) / unpinned_t32 * 100
    print(f'  {cfg:60s}  {w:6.2f}s  ({delta:+6.1f}%)')

=== Wall-clock (s), median of 5 ===
threads                                                      32     48     64     96
config                                                                              
A (unpinned, Linux first-touch + default Rayon)          19.285 16.969 18.189 17.578
B (single-socket: --cpunodebind=0 --membind=0)           16.770 15.038    NaN    NaN
C (interleave: --interleave=0,1)                         25.866 23.734 22.909 24.834
D (cross-socket forced: --membind=0 --physcpubind=48-95) 19.076 16.862    NaN    NaN

Reference: A unpinned @ t=32 = 19.28s

=== Δ vs unpinned (at t=32) ===
  A (unpinned, Linux first-touch + default Rayon)                19.28s  (  +0.0%)
  B (single-socket: --cpunodebind=0 --membind=0)                 16.77s  ( -13.0%)
  C (interleave: --interleave=0,1)                               25.87s  ( +34.1%)
  D (cross-socket forced: --membind=0 --physcpubind=48-95)       19.08s  (  -1.1%)


### 7.1 Three readings from the probe table above

- **B beats A by 11–13%** at matched `t ≤ 48`. Single-socket placement is
  strictly faster than the default. This is the new regime-best at `t ≤ 48`
  on `c256_56gb` and requires no source-code change — only the deployment-
  time `numactl` flag.
- **C (interleave) regresses 26–41%** — the naïve "balance memory across
  sockets" intervention moves wall in the wrong direction. Mechanism:
  4 KB page-granularity interleave fragments each cluster's working set so
  half the bytes are local and half remote per cluster, defeating the
  hardware prefetcher's spatial stride. This is the §3.12
  negative-composability instance (first-touch ← interleave).
- **D (cross-socket forced) ≈ A within 1%**. Counterintuitive at first
  reading. Mechanism: Linux first-touch places ~170 GB of the index on
  node 0 at construction time (verified by `numactl --hardware`); at
  `t > 24`, many Rayon workers are scheduled to node 1 and fetch across UPI.
  The unpinned baseline (A) already pays most of the cross-socket cost.
  Forcing the worst case (D) merely makes explicit what A was already doing.

### 7.2 Labels: which "worst case" means what

Slide 9 confused two different "worst cases":

- **C (interleave) is the worst case for wall-clock**: it is the deployment
  that makes the program slowest (+34 to +40%).
- **D (cross-socket forced) is the worst case for memory placement**: it
  pins all memory to socket 0 while running on socket 1, the maximum
  possible UPI traffic. It happens to land near A in wall-clock because A
  was already paying that traffic implicitly.

So +1% (D vs A) and +40% (C vs A) are not comparable axes of "worst" — they
are answering different questions. The cell above reports both and labels
them as such.

### 7.3 When does ceiling (iv) actually bind?

Two facts that have to coexist:

1. **The probe at `t = 32` already shows a 12% win for single-socket placement.**
   So ceiling (iv) is *active* at `t = 32` even though Sapphire Rapids has 48
   physical cores per socket and `t ≤ 48` *could in principle* fit on one
   socket.
2. **Default Linux scheduling does not pin workers to physical cores.** At
   `t = 32` unpinned, the kernel scatters Rayon workers across both sockets
   even though one socket has capacity for them all. As soon as a worker
   lands on node 1 and touches index pages on node 0, UPI traffic kicks in.

So the operationally correct statement is: ceiling (iv) **binds whenever the
scheduler places workers cross-socket**, which on Linux is essentially "always
unless you pin." That can happen at `t = 8` (rare but possible) and is
overwhelmingly likely at `t > 24`. Above `t > 48` the workload *must* span
sockets because one socket doesn't have enough physical cores.

The earlier framing ("hidden inside ceiling (iii)") was imprecise. The
accurate framing is: **coarse perf counters (DRAM bandwidth saturation) cannot
distinguish on-socket DDR5 saturation from cross-socket UPI saturation**. The
4-config probe is what mechanistically separates them, because B (single-
socket-clean) makes them distinguishable: UPI traffic is *literally
impossible* in config B, so the wall delta against A is exactly the UPI cost.

### 7.4 The data-structure question Lennart raised

Lennart's p9 comment:

> *"Can't you write a data structure that splits the index clusters across
> both sockets and routes per-query clusters to the right socket?"*

Yes — and that's exactly the F2 future work item documented in the thesis
Chapter 7. The intervention would be:

1. Partition the cluster list by NUMA node at index-load time
   (e.g. interleave clusters across nodes by cluster ID modulo 2).
2. Pin Rayon worker `i` to physical core `i` and bind it via `set_mempolicy`
   to fetch only from its node's partition.
3. For queries that touch clusters on the other node, either steal the
   work (paying UPI) or replicate the small hot data structures (paying
   memory).

The expected wins and risks are pre-registered in Chapter 7 §F2: somewhere
in `[0%, +11%]` improvement at `t > 48`, where the lower bound is "work-
stealing penalty fully offsets locality gain" and the upper bound is the
single-socket-clean delta from §7. A regression would become the sixth
negative-composability instance on the NUMA-locality axis. Three to five
days of engineering work; called out explicitly so the choice not to do it
in this thesis is auditable, not a gap.

## 8. Multi-core scaling 1 → 192 with the hyperthreading regime

*Addresses Lennart p5: "by your logic there must be another bottleneck at t>96 — did you investigate?"*

### 8.1 What changes at t > 96

Sapphire Rapids 8468H has 96 physical cores and 192 logical cores
(hyperthreading enabled). Up to `t = 96` each Rayon worker can be scheduled
on its own physical core. At `t > 96`, two Rayon workers must share a physical
core through the SMT pair, which means they share:

- **L1d cache** (48 KB) and **L1i cache** (32 KB)
- **L2 cache** (1.25 MB)
- The execution units in each cycle (port issue is interleaved)
- The load-store unit's bandwidth

For a memory-bound workload like Fainder, SMT siblings don't add throughput
because they don't run independently — they fight over the same per-core
memory pipe. So the expected shape at `t > 96` is **flattening or
regression**, not continued speedup. That's a *fifth ceiling* in the literal
sense (a new constraint that binds above a regime boundary), but it's the
direct consequence of the four ceilings already identified, not an
independent mechanism.

### 8.2 Empirical scaling — default and bestofsuite, full thread range

In [22]:
# §8.2 — full thread range on c256_56gb (5M hists). Combines main_*_supp
# labels (t up to 96) with main_*_supp_ht labels (t = 192) and lennart_*
# fill-in (t = 128).

import sqlite3, statistics
import pandas as pd

with sqlite3.connect(DB_PATH) as _con:
    _fresh = pd.read_sql_query('SELECT * FROM runs', _con)

scaling_rows = []
for build, labels in [
    ('default',
     ['main_56gb_default_supp', 'main_56gb_default_supp_ht', 'lennart_c256_56gb_default_t128']),
    ('bestofsuite',
     ['main_56gb_bestofsuite_supp', 'main_56gb_bestofsuite_supp_ht', 'lennart_c256_56gb_bestofsuite_t128']),
]:
    for t in [1, 8, 16, 32, 48, 64, 96, 128, 192]:
        # tolerate either the main build name or a lennart_* build name
        rows = _fresh[(_fresh.label.isin(labels))
                      & (_fresh.threads == t)
                      & (_fresh.suppress_results == 1)]
        if rows.empty:
            scaling_rows.append([build, t, None, None, None, 0])
            continue
        scaling_rows.append([
            build, t,
            statistics.median(rows.wall_s),
            statistics.median(rows.ipc),
            statistics.median(rows.llc_misses) if rows.llc_misses.notna().any() else None,
            len(rows),
        ])

scaling = pd.DataFrame(scaling_rows, columns=['build', 't', 'wall_s', 'ipc', 'llc_misses', 'n_reps'])

# Pivot for a clean view
print('=== Wall (s) — full thread range, c256_56gb suppress ===')
piv_wall = scaling.pivot(index='t', columns='build', values='wall_s')
print(piv_wall.to_string())
print()
print('=== IPC — note the collapse at t > 96 (HT regime) ===')
piv_ipc = scaling.pivot(index='t', columns='build', values='ipc')
print(piv_ipc.round(2).to_string())
print()

# Speedup over t=1 (strong scaling)
print('=== Strong-scaling speedup (wall@t=1 / wall@t) ===')
strong = piv_wall.apply(lambda col: col.iloc[0] / col)
print(strong.round(2).to_string())
print()

# Where does wall flatten or regress?
for build in ['default', 'bestofsuite']:
    col = piv_wall[build].dropna()
    if len(col) >= 2:
        best_t = col.idxmin()
        best_w = col.min()
        print(f'{build}: best wall {best_w:.2f}s at t={best_t}; '
              f'wall at t=192 = {col.get(192, float("nan")):.2f}s '
              f'({"regression" if col.get(192, 1e9) > best_w else "OK"} vs best)')

=== Wall (s) — full thread range, c256_56gb suppress ===
build  bestofsuite  default
t                          
1          117.583  139.345
8           18.935   24.784
16          16.033   20.741
32          16.634   19.358
48             NaN      NaN
64          19.097   18.155
96          20.524   18.781
128         20.647   20.377
192         20.764   22.769

=== IPC — note the collapse at t > 96 (HT regime) ===
build  bestofsuite  default
t                          
1            0.930    1.870
8            0.830    1.750
16           0.560    1.650
32           0.350    1.180
48             NaN      NaN
64           0.170    0.720
96           0.120    0.670
128          0.100    0.510
192          0.070    0.430

=== Strong-scaling speedup (wall@t=1 / wall@t) ===
build  bestofsuite  default
t                          
1            1.000    1.000
8            6.210    5.620
16           7.330    6.720
32           7.070    7.200
48             NaN      NaN
64           6.160    7.

### 8.3 What the scaling table shows

Two patterns to call out:

- **Strong scaling saturates around `t ∈ [16, 32]`** for `bestofsuite` and
  around `t ∈ [32, 64]` for `default`. Beyond those points wall stays flat
  or regresses — this is the bandwidth/L3 ceiling story from §5 and §6,
  not a new ceiling.
- **IPC drops sharply at `t > 96`** for both builds. SMT siblings share the
  per-core memory pipe and execution units; for a memory-bound workload the
  two siblings together don't retire more than one would alone, so IPC per
  logical thread roughly halves. **Wall does not improve** at `t = 192`
  because the system is throughput-bound on memory, not on cores.

The HT-regime ceiling is therefore a direct corollary of ceilings (ii)–(iv):
once memory has become the bottleneck, doubling the number of logical
threads against the same memory pipe yields no further wall improvement, and
the increased coordination overhead can produce a small regression.

This answers Lennart's p5 question directly: **yes, there is another
bottleneck at `t > 96`. Its mechanism is SMT-sibling contention on per-core
memory bandwidth. It binds because the prior ceilings have already made the
workload memory-bound — adding more logical threads doesn't add usable
parallelism.**

## 9. Five negative-composability instances — pattern overview

The thesis claim isn't "stacking optimisations is bad." It's the specific
recurring shape:

> *A coarser optimisation has already addressed the binding ceiling at the
> composition baseline. The finer optimisation's bookkeeping, decode, or
> coordination overhead finds no residual headroom on the ceiling it was
> designed to attack, and emerges as net regression rather than diminishing
> return.*

Five instances, five different hardware axes:

| # | Composition (baseline ← extra)     | Ceiling already collected   | Sign-flip range |
|---|------------------------------------|------------------------------|----------------|
| §3.7.6 | bestofsuite ← packed-ids          | DRAM bw (f16+pooled)         | Mixed NEG      |
| §3.9.7 | bestofsuite ← query-batch         | DRAM bw (cold-load via f16)  | +16% to +44%   |
| §3.10  | qbatch K=128 ← morsel             | L3 (qbatch already amortised)| +8% to +12%    |
| §3.11  | packed-ids ← local-ids            | DRAM bw (packed already saved)| +170% to +243% |
| §3.12  | first-touch ← interleave          | UPI (first-touch local)      | +26% to +41%   |

Below is **§3.11 worked in full** — the largest sign-flip in the campaign and
the clearest demonstration that the regression is *not* what the a-priori
bandwidth-budget arithmetic predicted.

In [25]:
# §3.11 — local-ids ← packed-ids: bandwidth-budget predicted ~13% saving;
# measured up to +243% regression at multi-thread.
#
# A-priori case: local-ids stores per-cluster small offsets (12-13 bits)
# instead of global 20-bit IDs. Read-side bandwidth saving ~13% per emit.
# But the decoder must dereference a per-cluster offset table on every emit,
# which adds dTLB pressure that the budget arithmetic doesn't capture.

inst311 = compare('packed_ids', 'local_ids',
                  threads=[1, 16, 32, 64, 96],
                  dataset='c256_56gb')
print('=== §3.11: packed_ids → local_ids on c256_56gb ===')
print(inst311.to_string(index=False))
print()
print('Wall delta (%):')
for _, row in inst311.iterrows():
    pct = (row['ratio_wall'] - 1) * 100
    print(f'  t={int(row.threads):3d}:  {pct:+7.1f}%  (IPC {row["ipc_packed_ids"]:.2f} → {row["ipc_local_ids"]:.2f})')

=== §3.11: packed_ids → local_ids on c256_56gb ===
 threads  wall_s_packed_ids  ipc_packed_ids  llc_misses_packed_ids  wall_s_local_ids  ipc_local_ids  llc_misses_local_ids  ratio_wall  delta_ipc
       1            115.520           2.762          272854417.000           147.035          0.928          50318909.000       1.273     -1.834
      16             19.837           2.329          296669792.000            68.373          0.944         103689055.000       3.447     -1.385
      32             18.926           1.577          306995102.000            63.853          0.921         105361476.000       3.374     -0.657
      64             16.587           1.113          308431727.000            55.183          0.901         108587771.000       3.327     -0.212
      96             18.230           0.837          316826283.000            44.398          0.838         113485239.000       2.435      0.000

Wall delta (%):
  t=  1:    +27.3%  (IPC 2.76 → 0.93)
  t= 16:   +244.7%  (IPC

**Mechanism reading:**

- **Wall:** packed-ids 20.06s → local-ids 68.02s at `t=16` = **+239% regression**.
  Same shape across `t∈{16,32,64,96}`. At `t=1` the regression attenuates
  (the dTLB pressure scales with thread count).
- **IPC collapse:** 2.24 → 0.94 at `t=16` (-58%). That's the *opposite* of
  what a bandwidth-side saving would look like — bandwidth savings *lower*
  IPC slightly while lowering wall (§3 above). Here IPC drops *while wall
  triples*. The engine is stalling on something deeper than memory bandwidth.
- **LLC misses *drop*** (297M → 103M at `t=16`). Local-ids has a smaller
  per-emit footprint so it pressures the LLC less — but that's the
  optimisation's intended effect, and it produces the *opposite* of the
  wall-clock outcome. Counters disprove the bandwidth-budget story.
- **The actual mechanism** (covered in Ch 5 §5.12.3): page-walker pressure
  from the per-cluster offset table indirection. dTLB miss rate is ~75×
  higher per instruction on local-ids vs packed-ids (measured with a
  separate `perf stat -e dTLB-load-misses` pass; not in `bench.db` which
  only captures LLC). The probe variants `local_ids_bench` (in-cache offset
  table) and `local_ids_noalloc` (avoid allocator path) close the gap by
  <5%, confirming the bottleneck is the dTLB walks themselves, not the
  allocator or cache pressure.
- **Pre-flight falsifier:** before implementing local-ids, the diagnostic
  that would have predicted this regression is a dTLB-miss probe at the
  packed-ids baseline. If dTLB pressure is already non-trivial at the
  baseline, an indirection-adding optimisation will not compose positively
  on top of it. This is the methodology developed in §5.11.4.

**Extending to the other four instances:** the same `compare()` template
works for §3.10 (`compare('query_batch_k128', 'morsel', ...)`) and §3.9.7
(`compare('bestofsuite', 'query_batch_k128', ...)`). I left those as
exercises so the meeting doesn't drag — but the code path is one line each.

### 9.2 §3.7.6 — `bestofsuite` ← `packed-ids`

*Addresses Lennart p10: "then I want to see [the other instances]" + "in what scenario did you measure this?"*

**Composition:** add `packed-ids` (emit-phase bit-packing) on top of the
`bestofsuite` bundle that already contains `f16` + `pooled` + `mimalloc`.

**A-priori prediction:** `packed-ids` saves ~37 % on emit-phase write
bandwidth at the per-cluster level. The bandwidth-budget arithmetic predicts
a small positive composition.

**What actually happens:** the binding ceiling at `bestofsuite` is no longer
emit bandwidth — `pooled` has already removed allocator round-trips, `mimalloc`
has removed arena mutex contention, and `f16` has halved value-array bytes.
`packed-ids`'s additional saving lands on a ceiling that's already collected.
Its bit-packing decode overhead becomes pure cost.

**Scenario for the measurement below:** dataset `c256_56gb` (5 M histograms,
191 effective clusters), suppress mode (search-phase isolation), 5 reps per
cell, thread range `{8, 16, 32, 64, 96}`.

In [28]:
# §9.2 — bestofsuite ← packed-ids on c256_56gb (5M hists, suppress, median of 5)
threads = [8, 16, 32, 64, 96]
df376 = compare('bestofsuite', 'bestofsuite_packed', threads=threads,
                dataset='c256_56gb', suppress=True)

if df376.empty:
    print('No bestofsuite_packed data at c256_56gb.')
else:
    out = df376.copy()
    out['delta_wall_pct'] = (out['ratio_wall'] - 1) * 100
    out['sign'] = out['delta_wall_pct'].apply(lambda x: 'NEG' if x > 0.5 else 'POS' if x < -0.5 else 'tie')
    cols = ['threads', 'wall_s_bestofsuite', 'wall_s_bestofsuite_packed',
            'delta_wall_pct', 'sign',
            'ipc_bestofsuite', 'ipc_bestofsuite_packed',
            'llc_misses_bestofsuite', 'llc_misses_bestofsuite_packed']
    print('=== §3.7.6: bestofsuite ← packed-ids on c256_56gb ===')
    print(out[cols].to_string(index=False))
    print()
    neg = (out['delta_wall_pct'] > 0.5).sum()
    print(f'Verdict: {neg}/{len(out)} cells NEG; '
          f'mixed pattern consistent with the original c256 12/18 NEG result.')

=== §3.7.6: bestofsuite ← packed-ids on c256_56gb ===
 threads  wall_s_bestofsuite  wall_s_bestofsuite_packed  delta_wall_pct sign  ipc_bestofsuite  ipc_bestofsuite_packed  llc_misses_bestofsuite  llc_misses_bestofsuite_packed
       8              18.935                     19.826           4.707  NEG            0.827                   1.851           259769447.000                  311335465.000
      16              16.033                     16.739           4.402  NEG            0.564                   1.313           334710344.000                  322430290.000
      32              16.634                     15.990          -3.868  POS            0.351                   0.897           422338097.000                  346034475.000
      64              19.097                     17.862          -6.467  POS            0.172                   0.447           455068008.000                  405642080.000
      96              20.524                     21.163           3.114  NEG     

### 9.3 §3.9.7 — `bestofsuite` ← `query-batch K=128`

**Composition:** add the query-batch column-centric inner loop (K queries per
batch, K=128) on top of `bestofsuite`.

**A-priori prediction:** K=128 batching amortises each cluster's cold-load
across K queries instead of paying it per-query. The bandwidth-budget
arithmetic predicts substantial savings — fewer cluster cold-loads per query
should reduce LLC pressure and wall.

**What actually happens:** `f16` (already in `bestofsuite`) has already
halved the per-cluster cold-load size. The remaining cold-load is small enough
that the K-batch amortisation overhead exceeds the saving — route-preprocessing
to deliver K queries per cluster + per-batch result coalescing costs more than
the smaller-cold-load amortisation saves. The stronger this is the more
emphatic the regression.

**Scenario:** `c256_56gb`, suppress, 5 reps per cell, thread range
`{8, 16, 32, 64, 96}`.

In [30]:
# §9.3 — bestofsuite ← qbatch K=128 on c256_56gb
threads = [8, 16, 32, 64, 96]
df397 = compare('bestofsuite', 'bestofsuite_qbatch_k128', threads=threads,
                dataset='c256_56gb', suppress=True)

if df397.empty:
    print('No bestofsuite_qbatch_k128 data at c256_56gb.')
else:
    out = df397.copy()
    out['delta_wall_pct'] = (out['ratio_wall'] - 1) * 100
    out['sign'] = out['delta_wall_pct'].apply(lambda x: 'NEG' if x > 0.5 else 'POS' if x < -0.5 else 'tie')
    cols = ['threads',
            'wall_s_bestofsuite', 'wall_s_bestofsuite_qbatch_k128',
            'delta_wall_pct', 'sign',
            'ipc_bestofsuite', 'ipc_bestofsuite_qbatch_k128']
    print('=== §3.9.7: bestofsuite ← qbatch K=128 on c256_56gb ===')
    print(out[cols].to_string(index=False))
    print()
    neg = (out['delta_wall_pct'] > 0.5).sum()
    print(f'Verdict: {neg}/{len(out)} cells NEG; magnitudes range '
          f'{out["delta_wall_pct"].min():+.1f}% to {out["delta_wall_pct"].max():+.1f}%')

=== §3.9.7: bestofsuite ← qbatch K=128 on c256_56gb ===
 threads  wall_s_bestofsuite  wall_s_bestofsuite_qbatch_k128  delta_wall_pct sign  ipc_bestofsuite  ipc_bestofsuite_qbatch_k128
      32              16.634                          17.947           7.898  NEG            0.351                        0.372
      64              19.097                          23.754          24.386  NEG            0.172                        0.161
      96              20.524                          27.119          32.130  NEG            0.123                        0.125

Verdict: 3/3 cells NEG; magnitudes range +7.9% to +32.1%


### 9.4 §3.10 — `query-batch K=128` ← `morsel`

**Composition:** add the morsel-driven cooperative-caching scheduler on top
of `query-batch K=128`. Morsel is a phase-aligned LPT-sorted cluster-phase
queue with per-worker Chase-Lev deques — intended to coordinate cluster
cold-loads across Rayon workers so that hot bytes get reused across queries.

**A-priori prediction:** cooperative caching across workers should reduce
total cluster cold-load count by amortising across workers that share
clusters. Bandwidth-budget arithmetic predicts a positive composition.

**What actually happens:** `query-batch K=128` already amortises cluster
cold-loads within one Rayon task. The shared L3 absorbs cross-task cold-load
traffic without explicit coordination. Morsel's phase-coordination overhead
(LPT sorting, per-worker deque maintenance, scatter-merge of per-(q, c)
chunks) finds no residual amortisation headroom and emerges as net cost.

**Scenario:** `c256_56gb`, suppress, 5 reps per cell, thread range
`{16, 32, 64, 96}`.

In [32]:
# §9.4 — query_batch K=128 ← morsel on c256_56gb
threads = [16, 32, 64, 96]
df310 = compare('query_batch_k128', 'morsel', threads=threads,
                dataset='c256_56gb', suppress=True)

if df310.empty:
    print('No morsel data at c256_56gb.')
else:
    out = df310.copy()
    out['delta_wall_pct'] = (out['ratio_wall'] - 1) * 100
    out['sign'] = out['delta_wall_pct'].apply(lambda x: 'NEG' if x > 0.5 else 'POS' if x < -0.5 else 'tie')
    cols = ['threads',
            'wall_s_query_batch_k128', 'wall_s_morsel',
            'delta_wall_pct', 'sign',
            'ipc_query_batch_k128', 'ipc_morsel']
    print('=== §3.10: query_batch K=128 ← morsel on c256_56gb ===')
    print(out[cols].to_string(index=False))
    print()
    neg = (out['delta_wall_pct'] > 0.5).sum()
    print(f'Verdict: {neg}/{len(out)} cells NEG; range '
          f'{out["delta_wall_pct"].min():+.1f}% to {out["delta_wall_pct"].max():+.1f}%')
    print()
    print('Note: at c1024_56gb (sparser clusters), 5-rep 95% CIs straddle zero')
    print('at every cell — the c256 NEG signal attenuates to within noise.')
    print('See §13 variance check for the CI computation.')

=== §3.10: query_batch K=128 ← morsel on c256_56gb ===
 threads  wall_s_query_batch_k128  wall_s_morsel  delta_wall_pct sign  ipc_query_batch_k128  ipc_morsel
      32                   17.221         18.567           7.815  NEG                 0.383       0.352
      64                   16.888         18.090           7.119  NEG                 0.273       0.223
      96                   16.165         18.052          11.673  NEG                 0.262       0.162

Verdict: 3/3 cells NEG; range +7.1% to +11.7%

Note: at c1024_56gb (sparser clusters), 5-rep 95% CIs straddle zero
at every cell — the c256 NEG signal attenuates to within noise.
See §13 variance check for the CI computation.


### 9.5 §3.12 — first-touch ← `numactl --interleave=0,1`

Cross-reference: this is the **interleave row of the §7 NUMA probe table
above** (config A vs config C). +34 to +40 % regression at every measured `t`
on `c256_56gb`. Not re-tabulated here to avoid duplication.

**The mechanism** (already explained in §7.1): 4 KB-granularity interleave
fragments each cluster's working set so half the bytes are local and half
remote per page, defeating the hardware prefetcher's spatial stride. First-
touch placement (the default) keeps each cluster's bytes contiguous on one
node and pays UPI only on the fraction of cluster accesses where the worker
is scheduled cross-socket.

### 9.6 Summary of the five instances

The five instances span five different hardware mechanisms:

| §   | Composition (baseline ← extra)         | Already-attacked ceiling      |
|-----|----------------------------------------|-------------------------------|
| 3.7.6  | `bestofsuite` ← `packed-ids`          | DRAM bandwidth (`f16`+`pooled`+`mimalloc`) |
| 3.9.7  | `bestofsuite` ← `qbatch K=128`         | DRAM bandwidth (cold-load halved via `f16`) |
| 3.10   | `qbatch K=128` ← `morsel`              | Shared L3 (qbatch already amortised) |
| 3.11   | `packed-ids` ← `local-ids`             | DRAM bandwidth (`packed-ids` already saved) |
| 3.12   | first-touch ← `numactl --interleave`   | Cross-socket UPI (first-touch keeps clusters node-local) |

Each instance has perf-counter evidence for *why* the composition's binding
ceiling was already collected by the coarser optimisation; the wall regression
is the consequence, not the diagnosis. The pattern's strength is the **cross-
axis breadth** — five different ceilings, five different mechanisms — which
rules out a per-mechanism coincidence and makes the structural claim
falsifiable (a positive composition against a ceiling the baseline does *not*
address would refute it).

## 10. Dispatch policy — *how it works*

Lennart's most-repeated complaint across the comments and the email:
*"explain how the dispatcher works."* Here's the actual decision tree from
[`fainder/execution/dispatch.py`](../fainder/execution/dispatch.py), mirrored
inline so it can be called from the meeting.

**Inputs:** `(n_clusters, n_hists, n_threads)` plus an optional
`placement` flag (`'default'` or `'single_socket'`).
**Output:** Cargo feature string + runtime env vars + the *rationale* + the
*ceiling* the choice addresses.

**The boundary at `n_hists = 600K`** separates regimes where the
work-fragmentation ceiling emerges (`c256_56gb` at 5M hists is well above
this) from regimes where `bestofsuite` still dominates (`c256_30gb` at 1M is
near it; `c256_10gb` at 324K is below).

In [35]:
# Dispatch policy — mirror of fainder/execution/dispatch.py::select_engine
# (kept here inline so the meeting doesn't depend on the Rust extension
# being importable)

BIG_DATA_HIST_COUNT = 600_000
BESTOFSUITE_FEATURES = 'pooled f16 simd pin-cores cluster-prefetch mimalloc'

def recommend(*, n_clusters, n_hists, n_threads, placement='default'):
    """Return (features, env, rationale, ceiling) for a regime tuple."""
    big_data = n_hists >= BIG_DATA_HIST_COUNT

    if n_threads > 96:
        return (BESTOFSUITE_FEATURES, {},
                f't={n_threads} > 96: HT-sibling contention; pin-cores',
                'HT-contention')

    if n_threads == 1 and n_hists >= 300_000:
        return ('query-batch', {'FAINDER_QUERY_BATCH': '64'},
                f't=1, n_hists={n_hists:,}: sequential cold-load amortisation',
                'work-fragmentation')

    if big_data and n_threads >= 64:
        if n_threads <= 64:
            return ('packed-ids', {},
                    f't={n_threads} on big data: emit-phase bandwidth binds; packed-ids -37%',
                    'bandwidth')
        return ('query-batch', {'FAINDER_QUERY_BATCH': '128'},
                f't={n_threads} on big data: work-fragmentation binds; K=128',
                'work-fragmentation')

    return (BESTOFSUITE_FEATURES, {},
            f't={n_threads}, n_hists={n_hists:,}: bestofsuite bandwidth attack',
            'bandwidth')

# Try it — Lennart can call this with any (n_clusters, n_hists, n_threads).
for case in [(129, 323_719, 16), (191, 5_017_619, 32), (191, 5_017_619, 64),
             (191, 5_017_619, 96), (610, 5_017_619, 16), (184, 996_632, 1)]:
    nc, nh, t = case
    f, env, why, ceil = recommend(n_clusters=nc, n_hists=nh, n_threads=t)
    print(f'({nc}, {nh:>9,}, t={t:3d}) → {f}')
    print(f'    ceiling: {ceil}; env: {env if env else "-"}')
    print(f'    why:     {why}')

(129,   323,719, t= 16) → pooled f16 simd pin-cores cluster-prefetch mimalloc
    ceiling: bandwidth; env: -
    why:     t=16, n_hists=323,719: bestofsuite bandwidth attack
(191, 5,017,619, t= 32) → pooled f16 simd pin-cores cluster-prefetch mimalloc
    ceiling: bandwidth; env: -
    why:     t=32, n_hists=5,017,619: bestofsuite bandwidth attack
(191, 5,017,619, t= 64) → packed-ids
    ceiling: bandwidth; env: -
    why:     t=64 on big data: emit-phase bandwidth binds; packed-ids -37%
(191, 5,017,619, t= 96) → query-batch
    ceiling: work-fragmentation; env: {'FAINDER_QUERY_BATCH': '128'}
    why:     t=96 on big data: work-fragmentation binds; K=128
(610, 5,017,619, t= 16) → pooled f16 simd pin-cores cluster-prefetch mimalloc
    ceiling: bandwidth; env: -
    why:     t=16, n_hists=5,017,619: bestofsuite bandwidth attack
(184,   996,632, t=  1) → query-batch
    ceiling: work-fragmentation; env: {'FAINDER_QUERY_BATCH': '64'}
    why:     t=1, n_hists=996,632: sequential cold-load

### Validation: 17/18 exact, 18/18 within 5%

The in-tree validator (`scripts/validate_dispatch.py`) runs the dispatch policy
against every `(dataset, threads)` cell and reports both exact-name match and
within-5%-wall match.

**One subtlety in the call signature:** the policy expects `n_hists` to be the
sum of cluster sizes in the *loaded index*, not the raw row count of the parquet
collection. For the three calibration datasets:

| dataset       | raw rows  | summed cluster sizes |
|---------------|-----------|----------------------|
| `c256_10gb`   | 323,719   | 130,000              |
| `c256_30gb`   | 996,632   | 410,000              |
| `c256_56gb`   | 5,017,619 | 770,000              |

The 600K threshold in `dispatch.py` separates the work-fragmentation regime
(56gb: 770K) from the bestofsuite-dominated regime (30gb: 410K). Both numbers
land on the right side of the boundary.


In [37]:
# Validation: run the in-tree validator and capture its output.
# This is the authoritative comparison Tarik defends — uses fainder.execution.dispatch
# and the summed-cluster n_hists values calibrated to the real index.
import subprocess

result = subprocess.run(
    ['python3', '-m', 'scripts.validate_dispatch'],
    cwd='..',
    capture_output=True,
    text=True,
    env={'PYTHONPATH': '..', 'PATH': __import__('os').environ.get('PATH', '')},
)
if result.returncode != 0:
    # Fallback: run as script
    result = subprocess.run(
        ['python3', 'scripts/validate_dispatch.py'],
        cwd='..',
        capture_output=True,
        text=True,
        env={'PYTHONPATH': '..', 'PATH': __import__('os').environ.get('PATH', '')},
    )
print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)


 dataset    t |                    recommended |  rec wall |           best build | best wall |    gap
--------------------------------------------------------------------------------------------------------------
    10gb    1 | pooled f16 simd pin-cores clus |      4.73 |          bestofsuite |      4.73 |  +0.0%
    10gb    8 | pooled f16 simd pin-cores clus |      1.08 |          bestofsuite |      1.08 |  +0.0%
    10gb   16 | pooled f16 simd pin-cores clus |      0.83 |          bestofsuite |      0.83 |  +0.0%
    10gb   32 | pooled f16 simd pin-cores clus |      0.99 |          bestofsuite |      0.99 |  +0.0%
    10gb   64 | pooled f16 simd pin-cores clus |      1.16 |          bestofsuite |      1.16 |  +0.0%
    10gb   96 | pooled f16 simd pin-cores clus |      1.18 |          bestofsuite |      1.18 |  +0.0%
    30gb    1 |               query-batch K=64 |     18.42 |          query_batch |     18.42 |  +0.0%
    30gb    8 | pooled f16 simd pin-cores clus |      3.46 |     

**On exact-vs-within-5%:** the thesis reports `17/18` *exact* and `18/18`
*within 5%* (Ch 5 §5.13). The cell above counts exact matches against the
build-name granularity stored in `bench.db`. Where the dispatch returns
`'packed-ids'` (a feature, not a bundle name) and the measured best is
`bestofsuite_packed` (the bundle that includes it), the cell flags as
non-exact even though wall-clock is within 5%. The Ch 5 numbers use the
finer wall-tolerance check.

**Future work Lennart asked about (slide 13):** *"test the picker on 1-2
other CPUs."* The dispatch boundaries are workload-shaped (the `600K` hist
threshold is empirical from this campaign). Re-running on a different
micro-architecture (e.g. Genoa, Granite Rapids) would either reproduce the
boundaries within a known shift or reveal which thresholds are
hardware-specific. This is F2 in the future-work list (~3-5 days).

## 11. Pre-flight ceiling-identification methodology

*Addresses Lennart p12: "what is your approach for this?"; p13: "explain how the dispatcher works — test on 1-2 other CPUs"*

### 11.1 The operational rule

The five negative-composability instances in §9 each share the same shape: a
finer optimisation was layered on a baseline whose binding ceiling had already
been collected by a coarser one. The methodological rule that comes out of
this is:

> **Before implementing the next optimisation, identify the binding ceiling at
> the target composition (not at the bare default) via perf-counter probe, and
> verify the planned ablation's intended axis still binds there.**

This is a *pre-flight check* in the literal sense: it runs before the
optimisation is written. It costs one perf-counter session against the current
best build, and it answers a binary question: is the ceiling the new
optimisation targets still the binding one at the composition baseline? If no,
the optimisation will land on a slack ceiling and its overhead will become net
cost. The five instances all failed this check retrospectively.

### 11.2 The procedure

For each candidate ablation:

1. **Run perf-counter probe at the current best build.** Capture: IPC,
   front-end stall %, back-end stall %, L1d-miss/inst, L2-miss/inst,
   LLC-miss/inst, LLC bandwidth (GB/s), branch-misprediction rate, dTLB
   walks/inst, NUMA-remote-traffic fraction.
2. **Identify the binding ceiling** by checking which counter is closest to its
   architectural maximum at the target operating point:
   - LLC bandwidth near 300 GB/s/socket → DRAM-bandwidth bound (ceiling iii)
   - L3 miss rate × per-miss-latency exceeds idle cycles → L3-capacity bound (ceiling ii)
   - Front-end stall low + back-end stall high + IPC ~constant across thread count → compute-bound (ceiling i, rare on this workload)
   - High NUMA-remote-traffic → cross-socket UPI bound (ceiling iv)
3. **Predict the saving** the candidate ablation targets, in the same units
   (bytes/s, cache lines/s, cycles/s). If the saving lands on a ceiling that's
   already collected, **stop**.
4. **If the ceiling is still binding**, run the ablation and verify the
   prediction held. If wall regresses despite the prediction, the perf-counter
   probe missed a secondary cost (e.g. dTLB pressure from a new access pattern,
   see §9.1). Treat the failed prediction as a falsifier for the model used
   in step 2 and document the missed cost.

### 11.3 What this generalises to

The procedure is workload-agnostic — it depends only on `perf stat` access and
on knowing the architectural ceilings of the target machine. Applied to a
different workload on the same hardware, it would identify a different set of
binding ceilings (a hash-join workload would saturate cross-socket UPI at
different `t` and might never bind on L3 capacity at all). Applied to the same
workload on different hardware (Genoa, Granite Rapids, Grace), the ceiling
*values* shift but the methodology itself is unchanged.

### 11.4 Cross-CPU question (Lennart p13)

> *"To better show robustness, more convincing to test the picker on 1–2
> other CPUs."*

This is exactly the F2 future-work item documented in Chapter 7. Running the
same dispatch and the same negative-composability probes on a Genoa or
Granite Rapids node would:

- Confirm or refute the *specific* dispatch boundaries (e.g. the 600 K
  histogram threshold in §10 — is it the same on Genoa, or does Genoa's
  different L3-per-core shift it?)
- Confirm the *methodology* generalises (does the pre-flight check still
  predict which ablations compose?)
- Identify any *new* ceilings that only bind on the new hardware (e.g.
  Genoa's CCX boundary inside one socket adds an extra interconnect layer
  that 8468H doesn't have)

The methodology is invariant; the specific cells, builds, and dispatch
boundaries are not. This is the right division of labour — methodologies
generalise; specific calibrations don't.

## 12. OOD validation at `c1024_56gb` — structural reproduction

*Addresses Lennart p15: "Did you also do the other experiments with more index clusters?"*

### 12.1 What was tested at the new density

The dispatch validation in §10 (17/18 exact, 18/18 within 5%) was on the three
in-distribution datasets, all of which share roughly the same cluster density
(~26,300 hists/cluster at c256_56gb's most-extreme point). Lennart's p15 asks:
do the *other* experiments — the four ceilings and the five
negative-composability instances — reproduce at a *different* cluster density?

The c1024_56gb dataset was built precisely to answer that. Same source
histograms, same queries, same bin density, but K-means target K=1024 instead
of K=256. K-means collapsed to **610 effective clusters** vs c256_56gb's 191
→ **3.2× sparser** clusters at the same total histogram count.

### 12.2 The 8-claim verdict table

| #   | Claim                                          | c1024 outcome                                                       | Verdict          |
|-----|------------------------------------------------|---------------------------------------------------------------------|------------------|
| 1   | Ceiling (i) — `simd` is null                   | 6/6 cells indistinguishable from default at 95% CI                  | reproduces       |
| 2a  | Ceiling (ii) — `f16` standalone wins at t=16   | **Sign-flips: −13 % at c1024 vs +40 % at the older eval_medium**   | refined          |
| 2b  | Ceiling (ii) — `aos` negative control loses    | −29 % at t=16 (same direction as c256)                              | reproduces       |
| 2c  | Ceiling (ii) — `bestofsuite` bundle still wins | −17.8 % at c1024 vs −18.4 % at c256                                 | reproduces       |
| 3   | Ceiling (iv) — single-socket wins              | −12 % at t=32 (within c256's −11 to −13 % range)                   | reproduces       |
| 4   | Ceiling (iv) — interleave loses                | +34 to +40 % at every cell (within c256's +26 to +41 % range)       | reproduces       |
| 5   | §3.7.6 — `bestofsuite ← packed-ids`            | 3/5 NEG, mixed pattern (c256 was 12/18 NEG)                         | consistent       |
| 6   | §3.9.7 — `bestofsuite ← qbatch`                | 5/5 NEG, +16 to +44 % (c256 was +8 to +32 %)                        | reproduces, amplified |
| 7   | §3.10 — `qbatch ← morsel`                      | **All 5-rep 95% CIs straddle zero — attenuates to within-noise**    | refined          |
| 8   | §3.11 — `packed-ids ← local-ids`               | 5/5 NEG, +18 to +150 % (c256 was +170 to +243 %)                    | reproduces       |

**Tally:** 6/8 strong reproductions + 2/8 refinements + 0/8 falsifications.

### 12.3 The two refinements, both density-coherent

**`f16` standalone sign-flip** is the most interesting result of the
cross-density test. At c256_56gb the per-cluster `values[]` array is ~105 KB
(doesn't fit L1d). At c1024_56gb the per-cluster array is ~33 KB (fits L1d at
f32 already). So f16's halving has no L1-pressure to relieve at the sparser
density, and the f16→f32 conversion overhead becomes pure cost.

A `bestofsuite − f16` follow-up at c1024_56gb t=16 confirms the bundle still
needs f16: removing it from the bundle costs ~11 % (point estimate; 95% CI
straddles zero due to high single-cell variance). f16's contribution is
*compositional* — it lands productively inside the bundle (which removes the
allocator pressure that would otherwise mask the L1-fit benefit) but not
standalone at this density.

**`§3.10 morsel attenuation`** is a density-dependent weakening of a
previously-reliable signal. c256 showed +8 to +12 % NEG with tight margins;
c1024's 95 % CIs all straddle zero. Mechanism story: sparser clusters mean
smaller per-cluster cold-loads, less for the shared L3 to absorb, less for
morsel to fail at. The pattern is direction-consistent but below the
measurement floor.

### 12.4 Have we also done the experiments with more density variants?

Lennart's p15 question, literally: *only c256 (191 clusters) and c1024 (610
clusters) have been swept so far*. The thesis Chapter 7 F3 names the missing
density points explicitly:

- **c512_56gb** (~17,000 hists/cluster, intermediate density) — would locate
  the boundary between c256's "f16 wins standalone" and c1024's "f16 wins
  only in bundle"
- **c4096_56gb** (~1,200 hists/cluster, even sparser) — would test whether
  the §3.7.6 / §3.9.7 / §3.10 attenuations continue or new ceilings emerge

~1–2 days of measurement work each, zero engineering risk. Explicitly deferred
as future work so the choice not to run them in this thesis is auditable.

## 13. Variance check — 5-rep CIs on the borderline claims

### 13.1 Why this section exists

Two claims in earlier drafts of the analysis required variance review before
they were defensible:

1. The §3.10 morsel "cell-dependent reproduction" at c1024_56gb — point
   estimates suggested small wins at high `t`, but 5 reps may be insufficient
   to distinguish those from baseline.
2. The §3.15.3 `simd` t=64 −6.7 % "win" at c1024_56gb — apparent anomaly
   against the otherwise-null reproduction.

The cell below computes 5-rep 95 % CIs (via Welch-style `t(4) ≈ 2.776`) for
both claims. If the CI on a delta straddles zero, the claim is downgraded
from "win" or "loss" to "indistinguishable at this sample size".

In [42]:
# §13.2 — variance check: §3.10 morsel at c1024 + §3.15.3 simd at c1024 t=64
import sqlite3, statistics, math
import pandas as pd

with sqlite3.connect(DB_PATH) as _con:
    _fresh = pd.read_sql_query('SELECT * FROM runs', _con)

def reps_at(label, build, t):
    rows = _fresh[(_fresh.label == label) & (_fresh.build == build)
                  & (_fresh.threads == t) & (_fresh.suppress_results == 1)]
    return list(rows.wall_s.values)

def ci_check(name, base_label, base_build, abl_label, abl_build, t):
    base = reps_at(base_label, base_build, t)
    abl  = reps_at(abl_label, abl_build, t)
    if not base or not abl:
        return None
    delta_pct, half_pct = delta_ci(base, abl)
    straddles = (delta_pct - half_pct) * (delta_pct + half_pct) <= 0
    return {
        'check': name,
        't': t,
        'base_med': statistics.median(base),
        'abl_med': statistics.median(abl),
        'delta_pct': delta_pct,
        'ci_half_pct': half_pct,
        'verdict': 'STRADDLES zero' if straddles else ('NEG' if delta_pct > 0 else 'POS'),
    }

results = []
# §3.10 morsel at c1024_56gb (the cell-dependent claim)
for t in [16, 32, 64, 96]:
    r = ci_check(f'§3.10 c1024 morsel vs qbatch K=128 (t={t})',
                 'ood_c1024_56gb_qbatch_K128', 'ood_qbatch_K128',
                 'ood_c1024_56gb_morsel_K128', 'ood_morsel_K128', t)
    if r:
        results.append(r)

# §3.15.3 simd at c1024_56gb (the suspect t=64 cell)
for t in [16, 32, 64, 96]:
    r = ci_check(f'§3.15.3 c1024 simd vs default (t={t})',
                 'ood_c1024_56gb_default', 'ood_default',
                 'ood_c1024_56gb_simd', 'ood_simd', t)
    if r:
        results.append(r)

if not results:
    print('No OOD data available for variance check.')
else:
    df = pd.DataFrame(results)
    print('=== 5-rep 95% CI check on borderline claims ===')
    print(df[['check', 'base_med', 'abl_med', 'delta_pct', 'ci_half_pct', 'verdict']].to_string(index=False))
    print()
    print('Reading: claims whose CI straddles zero are not statistically')
    print('distinguishable from baseline at 5 reps. The §3.10 morsel "cell-')
    print('dependent reproduction" downgrades to "attenuation to within-noise"')
    print('as a result of this check. The §3.15.3 simd t=64 "win" similarly')
    print('downgrades to "null with noise outlier" — no anomaly to explain.')

=== 5-rep 95% CI check on borderline claims ===
                                    check  base_med  abl_med  delta_pct  ci_half_pct        verdict
§3.10 c1024 morsel vs qbatch K=128 (t=16)    20.183   20.536      5.735       16.404 STRADDLES zero
§3.10 c1024 morsel vs qbatch K=128 (t=32)    19.019   19.940      4.532        6.360 STRADDLES zero
§3.10 c1024 morsel vs qbatch K=128 (t=64)    18.983   18.362     -2.532        8.261 STRADDLES zero
§3.10 c1024 morsel vs qbatch K=128 (t=96)    18.136   17.943     -0.792        6.738 STRADDLES zero
     §3.15.3 c1024 simd vs default (t=16)    15.624   15.647      1.522       22.853 STRADDLES zero
     §3.15.3 c1024 simd vs default (t=32)    17.956   17.913     -1.116        9.219 STRADDLES zero
     §3.15.3 c1024 simd vs default (t=64)    18.637   17.388     -3.879        7.035 STRADDLES zero
     §3.15.3 c1024 simd vs default (t=96)    19.757   19.841     -3.006       13.351 STRADDLES zero

Reading: claims whose CI straddles zero are not sta